In [2]:
%load_ext autoreload
%autoreload 2

# Tabel 011 kvantorid 2

Lisaandmetena kasutatakse skripriga 910 kokku kogutud lemma pos korpuses esinemise statistikat.

Tasakaalus korpusest kogutakse kokku tipud - ülemus + vahetu alluv, kus:
* ülemuse sõnaliik on `S` ja kääne üks nendest: (ad), (all), (abl), (ill), (adit), (in), (el), (tr)
* alluv eelneb lauses ülemusele;
* alluva sünrel on `nmod`, sõnaliik on `S` ja kääne (ad), (all), (abl), (ill), (adit), (in), (el), (tr)

**Ülesande originaalpüstitus**

2. ülemuse sõnaliik = S ja kääne = ad, all, abl, ill, adt, in, el või tr + alluva sünrel = nmod ja kääne = ad, all, abl, ill, adt, in, el või tr
   
Tulemuste tabelis võiksid olla järgmised veerud: alluva lemma, alluva kääne, alluva arv, ülemuse lemma, ülemuse kääne, ülemuse arv, kogu lause, ?päringule vastav fragment, alluva lemma koguarv korpuses.

**Tulemus**

Tulemuseks on tabel tsv formaadis.


Tabeli veerud
||||
|---|---|---|
|**child_lemma**| alluva lemma |---|
|**child_case**| alluva kääne |---|
|**child_number**| alluva arv |---|
|**parent_lemma**| ülemuse lemma |---|
|**parent_case**| ülemuse käänel |---|
|**parent_number**| ülemuse arv |---|
|**text**| ?päringule vastav fragment |---|
|**sentence**| teve lause tekst, kus ülemus ja alluv toodetud esile alakriipsudega  \_\_sõne\_\_ |---|
|**sentence_id**| lause id koondkorpuse andmebaasis|---|
|**child_lemma_total**| lemma + POS esinemise arv Tasakaalus korpuses |---|


In [3]:
from notebook_context import LISTS_FOLDER, corpus_reader

import pandas as pd
from datetime import datetime

FIELDNAMES = [
    "child_lemma",
    "child_pos",
    "child_case",
    "child_number",
    "parent_lemma",
    "parent_pos",
    "parent_case",
    "parent_number",
    "text",
    "sentence",
    "sentence_id",
    "child_lemma_total",
]

# UD cases: Ade (ad), All (all), Abl (abl), Ill (ill), Add (adit), Ine (in), Ela (el), Tra (tr)
FILTER_CASES = (
    "Ade",
    "All",
    "Abl",
    "Add",
    "Ill",
    "Ine",
    "Ela",
    "Tra",
)

LEMMAS_STAT = LISTS_FOLDER / "stats/lemmas.tsv"

TYPE = "quantifier_2"
date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

OUT_FILE = LISTS_FOLDER / "results" / f"{TYPE}_{date_time}.tsv"


In [4]:
# loeme sisse lemmade statistika ja teeme vastava dict
df_lemmas = pd.read_csv(LEMMAS_STAT, sep="\t", keep_default_na=False)
lemmas_stat = {f"{row['lemma']}\t{row['POS']}": int(row['total']) for _, row in df_lemmas.iterrows()}
lemmas_stat['olema\tVERB']

629

In [5]:
%%time

collocations = []
count = 0
for sentence_id, graph in corpus_reader.get_sentences():
    count += 1
    if not sentence_id:
        sentence_id = count

    # matrix for node distances
    dpath = graph.get_distances_matrix()

    # noun nodes
    noun_nodes = graph.get_nodes_by_attributes(attrname="POS", attrvalue="NOUN")

    # nmod
    nmod_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="nmod")

    # iteratsioon üle nimisõnade
    for noun in noun_nodes:
        noun_lemma = graph.nodes[noun]["lemma"]
        noun_pos = graph.nodes[noun]["POS"]
        noun_case = graph.get_node_case(noun)
        noun_number = graph.get_node_number(noun)

        if noun_case not in FILTER_CASES:
            continue
        # childnodes
        kids = [k for k in dpath[noun] if dpath[noun][k] == 1]

        # iterate over nmod children
        for nmod in nmod_nodes:
            # kui pole vahetu alluv, siis ei huvita
            if nmod not in kids:
                continue

            # nmod peab olema lauses enne ülemust
            if nmod > noun:
                continue
            nmod_case = graph.get_node_case(nmod)
            nmod_pos = graph.nodes[nmod]["POS"]
            if nmod_case not in FILTER_CASES:
                continue
            if nmod_pos != "NOUN":
                continue

            nmod_lemma = graph.nodes[nmod]["lemma"]
            nmod_number = graph.get_node_number(nmod)

            words = []
            for n in sorted(graph.nodes):
                if not n:
                    continue
                if n in (noun, nmod):
                    words.append(f'___{graph.nodes[n]["form"]}___')
                else:
                    words.append(graph.nodes[n]["form"])
            sentence_text = " ".join(words)

            text = " ".join([graph.nodes[n]["form"] for n in sorted((noun, nmod))])

            collocations.append(
                (
                    nmod_lemma,  # child_lemma
                    nmod_pos,  # child_pos
                    nmod_case,  # child_case
                    nmod_number,  # child_number
                    noun_lemma,  # parent_lemma
                    noun_pos,  # parent_pos
                    noun_case,  # parent_case
                    noun_number,  # parent_number
                    text,  # text
                    sentence_text,  # sentence
                    sentence_id,  # sentence_id
                    lemmas_stat.get(f"{nmod_lemma}\t{nmod_pos}", 0),  # child_lemma_total
                )
            )

print(f"sentences processed: {count}")
print(f"collocations found: {len(collocations)}")

../data/Model2Eesti-keele-kui-teise-keele-kooliõpikute-lausete-korpus-2021.conllu
sentences processed: 42279
collocations found: 233
CPU times: user 4.07 s, sys: 40.4 ms, total: 4.11 s
Wall time: 4.13 s


In [6]:

df = pd.DataFrame(collocations, columns=FIELDNAMES)
df.to_csv(OUT_FILE, sep="\t", index=None)
df.head()

,child_lemma,child_pos,child_case,child_number,parent_lemma,parent_pos,parent_case,parent_number,text,sentence,sentence_id,child_lemma_total
0,pere_naine,NOUN,Tra,Sing,minek,NOUN,Ela,Sing,perenaiseks minekust,Kui varem unistasin maale ___perenaiseks___ __...,264,12
1,perspektiivika,NOUN,Ela,Sing,tulevik,NOUN,Ela,Sing,perspektiivikast tulevikust,"On arusaadav , et noored unistavad ___perspekt...",362,3
2,kodu,NOUN,Ine,Sing,kook,NOUN,Ela,Plur,kodus kookidest,Ära ütle iialgi ära ___kodus___ küpsetatud ___...,434,714
3,enamik,NOUN,Ine,Sing,kiiver,NOUN,Ine,Plur,Enamikus kiivrites,___Enamikus___ sellistes ___kiivrites___ on kõ...,501,42
4,valge,NOUN,Ine,Plur,naiste_rõivas,NOUN,Ine,Plur,valgetes naisterõivastes,Need olid ___valgetes___ ___naisterõivastes___...,858,7
